---
numbering: false
---

# 3.5: Matrix multiplication

Something we've explored over the past few sections is the idea of performing two operations back to back. We could stretch a vector and then rotate it, for example. Can we describe the combined operation with just one matrix?

Yes! Applying two linear transformations back to back corresponds to **multiplying two matrices together**, which we will explore here.


In [1]:
from pathlib import Path
import sys
_notes_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'myst.yml').exists())
if str(_notes_root / 'ch03') not in sys.path:
    sys.path.insert(0, str(_notes_root / 'ch03'))
from friendly_visuals import mapping, transform, projection_line, projection_plane
from chapter3_visuals import save_figure, widget, show_3d


---

## The golden rule

Before we figure out how to multiply two matrices, let's check when we can.

:::{attention} Golden rule of matrix multiplication
The number of **columns in the first matrix** must equal the number of **rows in the second**:

$$\underbrace{A}_{n\times d}\;\underbrace{B}_{d\times p}
=\underbrace{AB}_{n\times p}.$$

The two inner numbers must match; the two outer numbers give the shape of the result.
:::

For example, a $2\times3$ matrix can multiply a $3\times4$ matrix, and the answer has shape $2\times4$. A $2\times3$ matrix cannot multiply a $2\times4$ matrix: the inner numbers, 3 and 2, don't match.

This is the same rule we used for matrix-vector multiplication. A vector is just a matrix with one column. Now the second matrix can have several columns.


---

## One column at a time

Here's the idea: **multiply the first matrix by each column of the second matrix, and put the answers next to each other.**

Let's try

$$A=\begin{bmatrix}2&-1\\1&3\end{bmatrix},\qquad
B=\begin{bmatrix}{\color{#3d81f6}1}&{\color{orange}4}\\{\color{#3d81f6}2}&{\color{orange}-1}\end{bmatrix}.$$

Both matrices have shape $2\times2$, so $AB$ also has shape $2\times2$. Start with the first column of $B$:

$$A{\color{#3d81f6}\begin{bmatrix}1\\2\end{bmatrix}}
=\begin{bmatrix}2(1)-1(2)\\1(1)+3(2)\end{bmatrix}
={\color{#3d81f6}\begin{bmatrix}0\\7\end{bmatrix}}.$$

Then do the same thing with the second column:

$$A{\color{orange}\begin{bmatrix}4\\-1\end{bmatrix}}
=\begin{bmatrix}2(4)-1(-1)\\1(4)+3(-1)\end{bmatrix}
={\color{orange}\begin{bmatrix}9\\1\end{bmatrix}}.$$

Place the two output vectors next to each other, keeping their order:

$$AB=\begin{bmatrix}{\color{#3d81f6}0}&{\color{orange}9}\\{\color{#3d81f6}7}&{\color{orange}1}\end{bmatrix}.$$

That's the whole calculation. We used matrix-vector multiplication twice. Click through the steps below to follow each column.


In [2]:
widget('matrix-product-widget.html', 'Multiply one column at a time')

:::{note} Definition: Matrix multiplication
If $A$ has shape $n\times d$ and the columns of a $d\times p$ matrix $B$ are $\vec b_1,\ldots,\vec b_p$, then

$$AB=\begin{bmatrix}|&|&&|\\A\vec b_1&A\vec b_2&\cdots&A\vec b_p\\|&|&&|\end{bmatrix}.$$

In words, **column $j$ of $AB$ is $A$ times column $j$ of $B$**.
:::

### A rectangular example

Let's try matrices that aren't square:

$$A=\begin{bmatrix}1&2&0\\-1&0&3\end{bmatrix},\qquad
B=\begin{bmatrix}{\color{#3d81f6}2}&{\color{orange}0}\\{\color{#3d81f6}1}&{\color{orange}-1}\\{\color{#3d81f6}-1}&{\color{orange}4}\end{bmatrix}.$$

The shapes are $(2\times3)(3\times2)$, so the answer is $2\times2$. The two columns are

$$A\begin{bmatrix}2\\1\\-1\end{bmatrix}=\begin{bmatrix}2+2\\-2-3\end{bmatrix}=\begin{bmatrix}4\\-5\end{bmatrix},\qquad
A\begin{bmatrix}0\\-1\\4\end{bmatrix}=\begin{bmatrix}-2\\12\end{bmatrix}.$$

Therefore,

$$AB=\begin{bmatrix}{\color{#3d81f6}4}&{\color{orange}-2}\\{\color{#3d81f6}-5}&{\color{orange}12}\end{bmatrix}.$$

### Finding one entry

Sometimes we only want one entry of the answer. To find the entry in row $i$, column $j$, multiply corresponding entries of **row $i$ of $A$** and **column $j$ of $B$**, then add.

For instance, the bottom-right entry above is

$$(-1)(0)+0(-1)+3(4)=12.$$

This is the dot product of the vectors $\begin{bmatrix}-1\\0\\3\end{bmatrix}$ and $\begin{bmatrix}0\\-1\\4\end{bmatrix}$. In general,

$$(AB)_{ij}=a_{i1}b_{1j}+\cdots+a_{id}b_{dj}.$$

Notice that we aren't just multiplying entries in matching positions. Every entry in the answer uses an entire row and an entire column.


---

## Two transformations

Now let's connect the calculation to our original question. Suppose we stretch horizontally by a factor of 2 and then rotate counterclockwise by $90^\circ$:

$$H=\begin{bmatrix}2&0\\0&1\end{bmatrix},\qquad
R=\begin{bmatrix}0&-1\\1&0\end{bmatrix}.$$

For an input $\begin{bmatrix}x\\y\end{bmatrix}$, the two steps are

$$\begin{bmatrix}x\\y\end{bmatrix}
\xrightarrow{\ H\ }\begin{bmatrix}2x\\y\end{bmatrix}
\xrightarrow{\ R\ }\begin{bmatrix}-y\\2x\end{bmatrix}.$$

The final rule has matrix $\begin{bmatrix}0&-1\\2&0\end{bmatrix}$. And that's exactly what we get by multiplying:

$$RH=\begin{bmatrix}0&-1\\1&0\end{bmatrix}\begin{bmatrix}2&0\\0&1\end{bmatrix}
=\begin{bmatrix}0&-1\\2&0\end{bmatrix}.$$

:::{important} Which matrix acts first?
$$\boxed{(AB)\vec x=A(B\vec x).}$$

The product $AB$ describes **first applying $B$, then applying $A$**. The matrix closest to the input acts first.
:::

That's why stretching first and rotating second gives $RH$. For example,

$$\begin{bmatrix}2\\1\end{bmatrix}\xrightarrow{\ H\ }\begin{bmatrix}4\\1\end{bmatrix}
\xrightarrow{\ R\ }\begin{bmatrix}-1\\4\end{bmatrix},\qquad
(RH)\begin{bmatrix}2\\1\end{bmatrix}=\begin{bmatrix}-1\\4\end{bmatrix}.$$

### Why the columns work

Suppose $B$ has columns $\vec b_1,\ldots,\vec b_p$. Then

$$\begin{aligned}
A(B\vec x)&=A(x_1\vec b_1+\cdots+x_p\vec b_p)\\
&=x_1A\vec b_1+\cdots+x_pA\vec b_p\\
&=(AB)\vec x.
\end{aligned}$$

We used linearity to move $A$ inside the sum. So the column-by-column rule gives exactly the matrix we need for doing two transformations back to back.

It also explains the golden rule: the output of $B$ must have the right number of entries to be an input to $A$.


---

## Order matters

Would rotating first and stretching second give the same result? Let's check:

$$HR=\begin{bmatrix}2&0\\0&1\end{bmatrix}\begin{bmatrix}0&-1\\1&0\end{bmatrix}
=\begin{bmatrix}0&-2\\1&0\end{bmatrix}.$$

This is different from $RH$. Our same input now follows

$$\begin{bmatrix}2\\1\end{bmatrix}\xrightarrow{\ R\ }\begin{bmatrix}-1\\2\end{bmatrix}
\xrightarrow{\ H\ }\begin{bmatrix}-2\\2\end{bmatrix}.$$

Why did the order matter? The stretch always doubles the **horizontal** coordinate. Rotating changes which part of the original vector points horizontally. Try both orders below.


In [3]:
widget('composition-widget.html', 'Stretch then rotate, or rotate then stretch')

**In general, $AB\ne BA$.** Unlike multiplication of real numbers, matrix multiplication is not commutative.

Sometimes reversing the order changes even the shape of the answer. For example, let

$$A=\begin{bmatrix}1&0\\0&1\\1&1\end{bmatrix},\qquad
B=\begin{bmatrix}1&0&0\\0&1&0\end{bmatrix}.$$

These are the two transformations at the end of Chapter 3.4. Multiplying gives

$$BA=\begin{bmatrix}1&0\\0&1\end{bmatrix},\qquad
AB=\begin{bmatrix}1&0&0\\0&1&0\\1&1&0\end{bmatrix}.$$

The first product has shape $2\times2$: append the sum of two entries, then discard it, and you're back where you started. The second has shape $3\times3$: discard the third entry, then replace it with the sum of the first two. You won't generally get the original third entry back.

And sometimes $AB$ is defined while $BA$ isn't. Try reversing shapes $2\times3$ and $3\times4$ to see why.

Some pairs of matrices do give the same answer in either order. For instance, doubling every coordinate and then rotating has the same effect as rotating and then doubling every coordinate.


---

## Properties

We can still use several familiar rules. Whenever the shapes allow the operations,

$$\begin{aligned}
(AB)C&=A(BC),\\
A(B+C)&=AB+AC,\\
(A+B)C&=AC+BC,\\
c(AB)&=(cA)B=A(cB).
\end{aligned}$$

The first property lets us change the parentheses, **keeping the matrices in the same order**. Both sides apply $C$ first, then $B$, then $A$. Changing the parentheses is different from reversing the order!

The identity matrix leaves vectors unchanged, so for a matrix $A$ of shape $n\times d$,

$$I_nA=A=AI_d.$$

### Projecting twice

Here's one last example. If $P=\begin{bmatrix}1&0\\0&0\end{bmatrix}$ projects onto the $x$-axis, then

$$P^2=\begin{bmatrix}1&0\\0&0\end{bmatrix}\begin{bmatrix}1&0\\0&0\end{bmatrix}
=\begin{bmatrix}1&0\\0&0\end{bmatrix}=P.$$

Does that make sense geometrically? After the first projection, our vector is already on the $x$-axis. Projecting it again leaves it there.

:::{tip} Activity: Practice multiplying
:class: dropdown

Let $A=\begin{bmatrix}1&1\\1&0\end{bmatrix}$ and $D=\begin{bmatrix}2&0\\0&1\end{bmatrix}$.

1. Compute $AD$ and $DA$, one column at a time. Are they equal?
2. For $\vec v=\begin{bmatrix}1\\2\end{bmatrix}$, compute $A(D\vec v)$ and $(AD)\vec v$. Do they agree?
3. Which product describes applying $A$ first and then stretching horizontally by a factor of 2?
:::
